In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os

import pandas as pd

os.chdir("/home/j/jackbarker/surface_predictor")
import pickle

import scripts.analyse_eofs as analyse_eofs
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def preprocess(ds):
    # Remove batch dimension if it exists
    if 'batch' in ds.dims:
        ds = ds.squeeze(dim="batch")
    return ds

def preprocess_era5_dataset(ds):
    ds = ds.rename({"valid_time": "datetime", "latitude": "lat", "longitude": "lon"})
    ds = ds.set_index({"datetime": "datetime"})
    ds = ds.reindex(lat=ds.lat[::-1])  # Reverse latitude order to match Graphcast data
    ds = ds.drop_vars(["number", "expver"])
    return ds

def average_area(start_date, end_date, tod, da):
    da = da.sel(lat=slice(2,16), lon=slice(32,45)).sel(datetime=slice(start_date, end_date))
    da = da.sel(datetime=da.datetime.dt.hour == tod)
    return da.mean(dim=["lat", "lon"])

In [ ]:
error_data_6am = xr.open_mfdataset(
    [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/errors_2016{day.strftime('%m%d')}-06_n0.nc" for day in pd.date_range("2016-01-01", "2016-12-31")],
    concat_dim="datetime",
    combine="nested",
    preprocess=preprocess,
)

In [ ]:
pred_data_6am = xr.open_mfdataset(
    [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/pred_2016{day.strftime('%m%d')}-06_n0.nc" for day in pd.date_range("2016-01-01", "2016-12-31")],
    concat_dim="datetime",
    combine="nested",
    preprocess=preprocess,
).rename({"time": "datetime"})
pred_data_6am = pred_data_6am.set_index({"datetime": "datetime"})

In [ ]:
pred_data_6am = pred_data_6am.set_index({"datetime": "datetime"})

In [ ]:
error_data_6am = error_data_6am.sel(lat=slice(0,20), lon=slice(30,50))
pred_data_6am = pred_data_6am.sel(lat=slice(0,20), lon=slice(30,50))

In [ ]:
era5_data_6am = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/localised_2mt_2016.nc")
era5_data_6am = era5_data_6am.rename({"valid_time": "datetime", "latitude": "lat", "longitude": "lon", "t2m": "2m_temperature"}).set_index({"datetime": "datetime"})
era5_data_6am = era5_data_6am.reindex(lat = era5_data_6am.lat[::-1])  # Reverse latitude order to match Graphcast data
era5_data_6am = era5_data_6am.drop_vars(["number", "expver"])

In [ ]:
def average_area(start_date, end_date, tod, da):
    da = da.sel(lat=slice(2,16), lon=slice(32,45)).sel(datetime=slice(start_date, end_date))
    da = da.sel(datetime=da.datetime.dt.hour == tod)
    return da.mean(dim=["lat", "lon"])

# Soil variables, evaporation and cloud cover

In [ ]:
src_evap = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/src_evap_2016.nc")
cc = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/cloud_cover_2016.nc")

cc = cc.reindex(latitude=cc.latitude[::-1])
src_evap = src_evap.reindex(latitude=src_evap.latitude[::-1])
cc = cc.sel(latitude=slice(0,20), longitude=slice(30,50))

all_data = xr.merge([src_evap, cc])

all_data = all_data.drop_vars(["number", "expver"])
all_data = all_data.rename(
    {
        "valid_time": "datetime",
        "longitude": "lon",
        "latitude": "lat",
    }
)


In [ ]:

timeseries_6am = average_area("2016-01-01", "2016-12-31", 6, all_data)
timeseries_midnight = average_area("2016-01-01", "2016-12-31", 0, all_data)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8))

axs[0,0].scatter(timeseries_6am["src"], error_2mt_series)
axs[0,0].set_xlabel("Skin resevoir content")
axs[0,0].set_ylabel("2m Temperature Error (K)")

axs[0,1].scatter(timeseries_6am["e"], error_2mt_series)
axs[0,1].set_xlabel("Evaporation")
axs[0,1].set_ylabel("2m Temperature Error (K)")

axs[1,0].scatter(timeseries_6am["evavt"], error_2mt_series)
axs[1,0].set_xlabel("Evaporation from vegetation")
axs[1,0].set_ylabel("2m Temperature Error (K)")

axs[1,1].scatter(timeseries_6am["tcc"], error_2mt_series)
axs[1,1].set_xlabel("Total Cloud Cover")
axs[1,1].set_ylabel("2m Temperature Error (K)")

plt.tight_layout()
plt.suptitle("Averaged temperature errors at 6am vs quantities at 6am (2016) over area (2-16N, 32-45E)", y=1.02)
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8))

axs[0,0].scatter(timeseries_midnight["src"], error_2mt_series)
axs[0,0].set_xlabel("Skin resevoir content")
axs[0,0].set_ylabel("2m Temperature Error (K)")

axs[0,1].scatter(timeseries_midnight["e"], error_2mt_series)
axs[0,1].set_xlabel("Evaporation")
axs[0,1].set_ylabel("2m Temperature Error (K)")

axs[1,0].scatter(timeseries_midnight["evavt"], error_2mt_series)
axs[1,0].set_xlabel("Evaporation from vegetation")
axs[1,0].set_ylabel("2m Temperature Error (K)")

axs[1,1].scatter(timeseries_midnight["tcc"], error_2mt_series)
axs[1,1].set_xlabel("Total Cloud Cover")
axs[1,1].set_ylabel("2m Temperature Error (K)")

plt.tight_layout()
plt.suptitle("Averaged temperature errors at 6am vs quantities at midnight (2016) over area (2-16N, 32-45E)", y=1.02)
plt.show()

# Fluxes

In [ ]:
flux_data = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/flux_2016.nc")
flux_data = flux_data.rename(
    {
        "slhf": "surface_latent_heat_flux",
        "sshf": "surface_sensible_heat_flux",
        "ssr": "surface_net_solar_radiation",
        "str": "surface_net_thermal_radiation",
        "latitude": "lat",
        "longitude": "lon",
        "valid_time": "datetime",
    }
)
flux_data = flux_data.drop_vars(["number", "expver"])
# reverse lat dimension
flux_data = flux_data.reindex(lat=flux_data.lat[::-1])
flux_data_timeseries_6am = average_area("2016-01-01", "2016-12-31", 6, flux_data)
flux_data_timeseries_midnight = average_area("2016-01-01", "2016-12-31", 0, flux_data)


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8))

axs[0,0].scatter(flux_data_timeseries_6am["surface_latent_heat_flux"], error_2mt_series)
axs[0,0].set_xlabel("Surface Latent Heat Flux (W/m^2)")
axs[0,0].set_ylabel("2m Temperature Error (K)")

axs[0,1].scatter(flux_data_timeseries_6am["surface_sensible_heat_flux"], error_2mt_series)
axs[0,1].set_xlabel("Surface Sensible Heat Flux (W/m^2)")
axs[0,1].set_ylabel("2m Temperature Error (K)")

axs[1,0].scatter(flux_data_timeseries_6am["surface_net_solar_radiation"], error_2mt_series)
axs[1,0].set_xlabel("Surface Net Solar Radiation (W/m^2)")
axs[1,0].set_ylabel("2m Temperature Error (K)")

axs[1,1].scatter(flux_data_timeseries_6am["surface_net_thermal_radiation"], error_2mt_series)
axs[1,1].set_xlabel("Surface Net Thermal Radiation (W/m^2)")
axs[1,1].set_ylabel("2m Temperature Error (K)")

plt.tight_layout()
plt.suptitle("Averaged temperature errors vs fluxes at 6am (2016) over area (2-16N, 32-45E)", y=1.02)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8))

axs[0,0].scatter(flux_data_timeseries_midnight["surface_latent_heat_flux"], error_2mt_series)
axs[0,0].set_xlabel("Surface Latent Heat Flux (W/m^2)")
axs[0,0].set_ylabel("2m Temperature Error (K)")

axs[0,1].scatter(flux_data_timeseries_midnight["surface_sensible_heat_flux"], error_2mt_series)
axs[0,1].set_xlabel("Surface Sensible Heat Flux (W/m^2)")
axs[0,1].set_ylabel("2m Temperature Error (K)")

axs[1,0].scatter(flux_data_timeseries_midnight["surface_net_solar_radiation"], error_2mt_series)
axs[1,0].set_xlabel("Surface Net Solar Radiation (W/m^2)")
axs[1,0].set_ylabel("2m Temperature Error (K)")

axs[1,1].scatter(flux_data_timeseries_midnight["surface_net_thermal_radiation"], error_2mt_series)
axs[1,1].set_xlabel("Surface Net Thermal Radiation (W/m^2)")
axs[1,1].set_ylabel("2m Temperature Error (K)")

plt.tight_layout()
plt.suptitle("Averaged temperature errorsa at 2am vs fluxes at midnight (2016) over area (2-16N, 32-45E)", y=1.02)

# Precipitation

In [ ]:
with open("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/time_series_data/basic_era5_series_6am.pickle", "rb") as f:
    all_era5_series_6am = pickle.load(f)

plt.scatter(all_era5_series_6am["total_precipitation_6hr"], error_2mt_series)
plt.xlabel("ERA5 6hr Total Precipitation (m)")
plt.ylabel("2m Temperature Error (K)")
plt.title("Total Precipitation vs 2m Temperature Error at 6am (2016) over area (2-16N, 32-45E)")

# Ensemble spreads

In [ ]:
spreads = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/temp_spreads.nc")
spreads = spreads.rename(
    {
        "valid_time": "datetime",
        "longitude": "lon",
        "latitude": "lat",
    }
)
spreads = spreads.drop_vars(["number", "expver"])
spreads = spreads.reindex(lat=spreads.lat[::-1])

spreads_series_6am = average_area("2016-01-01", "2016-12-31", 6, spreads)
spreads_series_3am = average_area("2016-01-01", "2016-12-31", 3, spreads)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8))

for i, ax in enumerate(axs.flat[:-1]):
    ax.scatter(spreads_series_6am.isel(pressure_level=i)["t"], error_2mt_series)
    ax.set_xlabel(f"Ensemble Spread at {spreads.pressure_level[i].values} hPa")
    ax.set_ylabel("2m Temperature Error (K)")

plt.tight_layout()
plt.suptitle("Averaged temperature errors at 6am vs ensemble spreads (2016) over area (2-16N, 32-45E)", y=1.02)
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8))

for i, ax in enumerate(axs.flat[:-1]):
    ax.scatter(spreads_series_3am.isel(pressure_level=i)["t"], error_2mt_series)
    ax.set_xlabel(f"Ensemble Spread at {spreads.pressure_level[i].values} hPa")
    ax.set_ylabel("2m Temperature Error (K)")

plt.tight_layout()
plt.suptitle("Averaged temperature errors at 6am vs ensemble spreads at 3am (2016) over area (2-16N, 32-45E)", y=1.02)
plt.show()

# Snow

In [ ]:
africa_snow = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/africa_snow_2016.nc")

africa_snow = africa_snow.rename(
    {
        "valid_time": "datetime",
        "longitude": "lon",
        "latitude": "lat",
    }
)
africa_snow = africa_snow.drop_vars(["number", "expver"])
africa_snow = africa_snow.reindex(lat=africa_snow.lat[::-1])

africa_snow_series_6am = average_area("2016-01-01", "2016-12-31", 6, africa_snow)
africa_snow_series_3am = average_area("2016-01-01", "2016-12-31", 3, africa_snow)

In [ ]:
plt.scatter(localised_snow_series_6am["sd"], error_2mt_series)

# Looking for the explicit convection conditions

In [ ]:
with open("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/time_series_data/basic_era5_series_6am.pickle", "rb") as f:
    tsdata = pickle.load(f)

tsdata

In [ ]:
era5_data_6am = pred_data_6am - error_data_6am

uwind_data = era5_data_6am["10m_u_component_of_wind"].load()
vwind_data = era5_data_6am["10m_v_component_of_wind"].load()
humid_data = era5_data_6am["specific_humidity"].sel(level=1000).load()

In [ ]:
# set up 3 plots with u wind, v wind and humidity on a chosen date
fig, axs = plt.subplots(3, 1, figsize=(10, 15), subplot_kw={'projection': ccrs.PlateCarree()})

time = "2016-01-18T06:00:00"
axs[0].set_title("10m U Component of Wind")

uwind_data.sel(datetime=time).plot(ax=axs[0], transform=ccrs.PlateCarree(), cmap='coolwarm', cbar_kwargs={'label': 'U Wind (m/s)'})
axs[0].coastlines()
axs[0].add_feature(cfeature.BORDERS, linestyle=':')
axs[0].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())

axs[1].set_title("10m V Component of Wind")
vwind_data.sel(datetime=time).plot(ax=axs[1], transform=ccrs.PlateCarree(), cmap='coolwarm', cbar_kwargs={'label': 'V Wind (m/s)'})
axs[1].coastlines()
axs[1].add_feature(cfeature.BORDERS, linestyle=':')
axs[1].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())

axs[2].set_title("Specific Humidity at 1000 hPa")
humid_data.sel(datetime=time).plot(ax=axs[2], transform=ccrs.PlateCarree(), cmap='viridis', cbar_kwargs={'label': 'Specific Humidity (kg/kg)'})
axs[2].coastlines()
axs[2].add_feature(cfeature.BORDERS, linestyle=':')
axs[2].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())

plt.tight_layout()
plt.show()



In [ ]:
# Create a figure with multiple consecutive days
dates = ["2016-01-18T06:00:00", "2016-01-19T06:00:00", "2016-01-20T06:00:00", "2016-01-21T06:00:00"]
n_days = len(dates)

fig, axs = plt.subplots(3, n_days, figsize=(5*n_days, 15), subplot_kw={'projection': ccrs.PlateCarree()})

for day_idx, time in enumerate(dates):
    # U Wind
    axs[0, day_idx].set_title(f"10m U Wind - {time[:10]}")
    uwind_data.sel(datetime=time).plot(
        ax=axs[0, day_idx], 
        transform=ccrs.PlateCarree(), 
        cmap='coolwarm', 
        cbar_kwargs={'label': 'U Wind (m/s)'},
        add_colorbar=True
    )
    axs[0, day_idx].coastlines()
    axs[0, day_idx].add_feature(cfeature.BORDERS, linestyle=':')
    axs[0, day_idx].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())
    
    # V Wind
    axs[1, day_idx].set_title(f"10m V Wind - {time[:10]}")
    vwind_data.sel(datetime=time).plot(
        ax=axs[1, day_idx], 
        transform=ccrs.PlateCarree(), 
        cmap='coolwarm', 
        cbar_kwargs={'label': 'V Wind (m/s)'},
        add_colorbar=True
    )
    axs[1, day_idx].coastlines()
    axs[1, day_idx].add_feature(cfeature.BORDERS, linestyle=':')
    axs[1, day_idx].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())
    
    # Specific Humidity
    axs[2, day_idx].set_title(f"Specific Humidity - {time[:10]}")
    humid_data.sel(datetime=time).plot(
        ax=axs[2, day_idx], 
        transform=ccrs.PlateCarree(), 
        cmap='viridis', 
        cbar_kwargs={'label': 'Specific Humidity (kg/kg)'},
        add_colorbar=True
    )
    axs[2, day_idx].coastlines()
    axs[2, day_idx].add_feature(cfeature.BORDERS, linestyle=':')
    axs[2, day_idx].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())

plt.tight_layout()
plt.suptitle("Meteorological Conditions at 6am - Consecutive Days", y=0.98, fontsize=16)
plt.show()

In [ ]:
# Create a figure with multiple consecutive days including 2mt errors
start_date = "2016-01-15"
end_date = "2016-01-21"
dates = [f"{day.strftime('%Y-%m-%d')}T06:00:00" for day in pd.date_range(start_date, end_date)]
n_days = len(dates)

fig, axs = plt.subplots(4, n_days, figsize=(5*n_days, 20), subplot_kw={'projection': ccrs.PlateCarree()})

for day_idx, time in enumerate(dates):
    # U Wind
    axs[0, day_idx].set_title(f"10m U Wind - {time[:10]}")
    uwind_data.sel(datetime=time).plot(
        ax=axs[0, day_idx], 
        transform=ccrs.PlateCarree(), 
        cmap='coolwarm', 
        cbar_kwargs={'label': 'U Wind (m/s)'},
        add_colorbar=True
    )
    axs[0, day_idx].coastlines()
    axs[0, day_idx].add_feature(cfeature.BORDERS, linestyle=':')
    axs[0, day_idx].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())
    axs[0, day_idx].set_title(f"10m U Wind - {time[:10]}")
    
    # V Wind
    axs[1, day_idx].set_title(f"10m V Wind - {time[:10]}")
    vwind_data.sel(datetime=time).plot(
        ax=axs[1, day_idx], 
        transform=ccrs.PlateCarree(), 
        cmap='coolwarm', 
        cbar_kwargs={'label': 'V Wind (m/s)'},
        add_colorbar=True
    )
    axs[1, day_idx].coastlines()
    axs[1, day_idx].add_feature(cfeature.BORDERS, linestyle=':')
    axs[1, day_idx].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())
    axs[1, day_idx].set_title(f"10m V Wind - {time[:10]}")
    
    # Specific Humidity
    axs[2, day_idx].set_title(f"Specific Humidity - {time[:10]}")
    humid_data.sel(datetime=time).plot(
        ax=axs[2, day_idx], 
        transform=ccrs.PlateCarree(), 
        cmap='viridis', 
        cbar_kwargs={'label': 'Specific Humidity (kg/kg)'},
        add_colorbar=True
    )
    axs[2, day_idx].coastlines()
    axs[2, day_idx].add_feature(cfeature.BORDERS, linestyle=':')
    axs[2, day_idx].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())
    axs[2, day_idx].set_title(f"Specific Humidity - {time[:10]}")
    
    # 2m Temperature Error
    axs[3, day_idx].set_title(f"2m Temperature Error - {time[:10]}")
    error_data_6am["2m_temperature"].sel(datetime=time).plot(
        ax=axs[3, day_idx], 
        transform=ccrs.PlateCarree(), 
        cmap='RdBu_r', 
        cbar_kwargs={'label': '2m Temperature Error (K)'},
        add_colorbar=True
    )
    axs[3, day_idx].coastlines()
    axs[3, day_idx].add_feature(cfeature.BORDERS, linestyle=':')
    axs[3, day_idx].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())
    axs[3, day_idx].set_title(f"2m Temperature Error - {time[:10]}")

plt.tight_layout()
plt.show()

In [ ]:
# Create a figure with multiple consecutive days including 2mt errors
start_date = "2016-01-15"
end_date = "2016-01-21"
dates = [f"{day.strftime('%Y-%m-%d')}T06:00:00" for day in pd.date_range(start_date, end_date)]
n_days = len(dates)

fig, axs = plt.subplots(4, n_days, figsize=(5*n_days + 2, 20), subplot_kw={'projection': ccrs.PlateCarree()})

# Calculate symmetric min/max for each variable
uwind_data_subset = uwind_data.sel(datetime=dates)
vwind_data_subset = vwind_data.sel(datetime=dates)
humid_data_subset = humid_data.sel(datetime=dates)
error_data_subset = error_data_6am["2m_temperature"].sel(datetime=dates)

uwind_max = 10
vwind_max = 12
humid_min = 0
humid_max = 0.02
error_max = 5

for day_idx, time in enumerate(dates):
    # U Wind
    
    uwind_data.sel(datetime=time).plot(
        ax=axs[0, day_idx], 
        transform=ccrs.PlateCarree(), 
        cmap='coolwarm', 
        vmin=-uwind_max, 
        vmax=uwind_max,
        add_colorbar=False
    )
    axs[0, day_idx].set_title(f"10m U Wind - {time[:10]} 0600UTC")
    axs[0, day_idx].coastlines()
    axs[0, day_idx].add_feature(cfeature.BORDERS, linestyle=':')
    axs[0, day_idx].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())
    
    # V Wind
    
    vwind_data.sel(datetime=time).plot(
        ax=axs[1, day_idx], 
        transform=ccrs.PlateCarree(), 
        cmap='coolwarm', 
        vmin=-vwind_max, 
        vmax=vwind_max,
        add_colorbar=False
    )
    axs[1, day_idx].set_title(f"10m V Wind - {time[:10]} 0600UTC")
    axs[1, day_idx].coastlines()
    axs[1, day_idx].add_feature(cfeature.BORDERS, linestyle=':')
    axs[1, day_idx].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())
    
    # Specific Humidity
    
    humid_data.sel(datetime=time).plot(
        ax=axs[2, day_idx], 
        transform=ccrs.PlateCarree(), 
        cmap='viridis', 
        vmin=humid_min, 
        vmax=humid_max,
        add_colorbar=False
    )
    axs[2, day_idx].set_title(f"Specific Humidity - {time[:10]} 0600UTC")
    axs[2, day_idx].coastlines()
    axs[2, day_idx].add_feature(cfeature.BORDERS, linestyle=':')
    axs[2, day_idx].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())
    
    # 2m Temperature Error
    
    error_data_6am["2m_temperature"].sel(datetime=time).plot(
        ax=axs[3, day_idx], 
        transform=ccrs.PlateCarree(), 
        cmap='RdBu_r', 
        vmin=-error_max, 
        vmax=error_max,
        add_colorbar=False
    )
    axs[3, day_idx].set_title(f"2mT Error - {time[:10]} 0600UTC")
    axs[3, day_idx].coastlines()
    axs[3, day_idx].add_feature(cfeature.BORDERS, linestyle=':')
    axs[3, day_idx].set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())

# Add shared vertical colorbars on the right side
import matplotlib as mpl

# U Wind colorbar
norm_uwind = mpl.colors.Normalize(vmin=-uwind_max, vmax=uwind_max)
sm_uwind = plt.cm.ScalarMappable(cmap='coolwarm', norm=norm_uwind)
sm_uwind.set_array([])
cbar_uwind = fig.colorbar(sm_uwind, ax=axs[0, :], orientation='vertical', 
                         fraction=0.02, pad=0.15, aspect=10)
cbar_uwind.set_label('U Wind (m/s)')

# V Wind colorbar
norm_vwind = mpl.colors.Normalize(vmin=-vwind_max, vmax=vwind_max)
sm_vwind = plt.cm.ScalarMappable(cmap='coolwarm', norm=norm_vwind)
sm_vwind.set_array([])
cbar_vwind = fig.colorbar(sm_vwind, ax=axs[1, :], orientation='vertical', 
                         fraction=0.02, pad=0.15, aspect=10)
cbar_vwind.set_label('V Wind (m/s)')

# Humidity colorbar (non-symmetric)
norm_humid = mpl.colors.Normalize(vmin=humid_min, vmax=humid_max)
sm_humid = plt.cm.ScalarMappable(cmap='viridis', norm=norm_humid)
sm_humid.set_array([])
cbar_humid = fig.colorbar(sm_humid, ax=axs[2, :], orientation='vertical', 
                         fraction=0.02, pad=0.15, aspect=10)
cbar_humid.set_label('Specific Humidity (kg/kg)')

# Error colorbar
norm_error = mpl.colors.Normalize(vmin=-error_max, vmax=error_max)
sm_error = plt.cm.ScalarMappable(cmap='RdBu_r', norm=norm_error)
sm_error.set_array([])
cbar_error = fig.colorbar(sm_error, ax=axs[3, :], orientation='vertical', 
                         fraction=0.02, pad=0.15, aspect=10)
cbar_error.set_label('2m Temperature Error (K)')

# plt.tight_layout()
plt.show()

# Raw Graphcast and ERA5 values

In [ ]:
error_series = average_area("2016-01-01", "2016-12-31", 6, error_data_6am["2m_temperature"]).load()
gc_series = average_area("2016-01-01", "2016-12-31", 6, pred_data_6am["2m_temperature"]).load()
era5_series = average_area("2016-01-01", "2016-12-31", 6, era5_data_6am["2m_temperature"]).load()

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 15))

error_times = error_series.datetime[np.abs(error_series) > 1]

plt.suptitle("6am averaged 2m Temperature over area (2-16N, 32-45E) in 2016", fontsize=16)

axs[0].plot(error_series.datetime, error_series, label="Graphcast Error")
axs[0].set_title("Graphcast 2m Temperature Errors (Graphcast - ERA5)")
axs[0].set_xlabel("Date")
axs[0].set_ylabel("Temperature Error (K)")

axs[1].plot(gc_series.datetime, gc_series, label="Graphcast Prediction")
axs[1].set_title("Graphcast 2m Temperature Prediction")
axs[1].set_xlabel("Date")
axs[1].set_ylabel("Temperature (K)")

axs[2].plot(era5_series.datetime, era5_series, label="ERA5 2m Temperature")
axs[2].set_title("ERA5 2m Temperature")
axs[2].set_xlabel("Date")
axs[2].set_ylabel("Temperature (K)")

for i in range(3):
    # plot vertical lines at each error time
    for error_time in error_times:
        axs[i].axvline(error_time.to_numpy(), color='red', linestyle='--', alpha=0.5)

plt.show()


In [ ]:
error_series_t800 = average_area("2016-01-01", "2016-12-31", 6, error_data_6am["temperature"].sel(level=800)).load()
gc_series_t800 = average_area("2016-01-01", "2016-12-31", 6, pred_data_6am["temperature"].sel(level=800)).load()
era5_series_t800 = average_area("2016-01-01", "2016-12-31", 6, era5_data_6am["temperature"].sel(level=800)).load()

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 15))

# error_times = error_series.datetime[np.abs(error_series) > 1]

plt.suptitle("6am averaged 800hPa Temperature over area (2-16N, 32-45E) in 2016", fontsize=16)

axs[0].plot(error_series_t800.datetime, error_series_t800, label="Graphcast Error")
axs[0].set_title("Graphcast 800hPa Temperature Errors (Graphcast - ERA5)")
axs[0].set_xlabel("Date")
axs[0].set_ylabel("Temperature Error (K)")

axs[1].plot(gc_series_t800.datetime, gc_series_t800, label="Graphcast Prediction")
axs[1].set_title("Graphcast 800hPa Temperature Prediction")
axs[1].set_xlabel("Date")
axs[1].set_ylabel("Temperature (K)")

axs[2].plot(era5_series_t800.datetime, era5_series_t800, label="ERA5")
axs[2].set_title("ERA5 800hPa Temperature")
axs[2].set_xlabel("Date")
axs[2].set_ylabel("Temperature (K)")

# for i in range(3):
#     # plot vertical lines at each error time
#     for error_time in error_times:
#         axs[i].axvline(error_time.to_numpy(), color='red', linestyle='--', alpha=0.5)

plt.show()


In [ ]:
error_series_q800 = average_area("2016-01-01", "2016-12-31", 6, error_data_6am["specific_humidity"].sel(level=800)).load()
gc_series_q800 = average_area("2016-01-01", "2016-12-31", 6, pred_data_6am["specific_humidity"].sel(level=800)).load()
era5_series_q800 = average_area("2016-01-01", "2016-12-31", 6, era5_data_6am["specific_humidity"].sel(level=800)).load()

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 15))

# error_times = error_series.datetime[np.abs(error_series) > 1]

plt.suptitle("6am averaged 800hPa Specific Humidity over area (2-16N, 32-45E) in 2016", fontsize=16)

axs[0].plot(error_series_q800.datetime, error_series_q800, label="Graphcast Error")
axs[0].set_title("Graphcast 800hPa Specific Humidity Errors (Graphcast - ERA5)")
axs[0].set_xlabel("Date")
axs[0].set_ylabel("Specific Humidity Error")

axs[1].plot(gc_series_q800.datetime, gc_series_q800, label="Graphcast Prediction")
axs[1].set_title("Graphcast 800hPa Specific Humidity Prediction")
axs[1].set_xlabel("Date")
axs[1].set_ylabel("Specific Humidity")

axs[2].plot(era5_series_q800.datetime, era5_series_q800, label="ERA5")
axs[2].set_title("ERA5 800hPa Specific Humidity")
axs[2].set_xlabel("Date")
axs[2].set_ylabel("Specific Humidity")


# for i in range(3):
#     # plot vertical lines at each error time
#     for error_time in error_times:
#         axs[i].axvline(error_time.to_numpy(), color='red', linestyle='--', alpha=0.5)

plt.show()


# Statisical comparison of fields on error vs non-error days

In [ ]:
# Build a dataset containing all non-anomaly temperature series (6am, area-mean, 2016)
# This collects: ERA5/GC/ERROR at 2m, ERA5 skin temperature, ERA5 pressure-level temps, and 800 hPa series when available.

# Align core 2m and skin temperature series on common datetime index
era5_2m, gc_2m, err_2m, skt = xr.align(era5_series, gc_series, error_series, skint_time_series, join='inner')

# Align pressure-level ERA5 temperatures to the same datetimes
try:
    t_levels = temp_time_series.sel(datetime=era5_2m.datetime)
except Exception:
    # Fallback to inner join alignment if needed
    t_levels, era5_2m = xr.align(temp_time_series, era5_2m, join='inner')

# Create the dataset
temp_series_ds = xr.Dataset()

temp_series_ds['2m_temperature_era5'] = era5_2m
temp_series_ds['2m_temperature_gc'] = gc_2m
temp_series_ds['2m_temperature_error'] = err_2m

# Skin temperature (ERA5)
temp_series_ds['skin_temperature_era5'] = skt

# ERA5 temperature across pressure levels (dims: datetime, pressure_level)
temp_series_ds['temperature_era5'] = t_levels



# Add metadata
temp_series_ds.attrs.update({
    'description': 'Area-mean (2–16N, 32–45E) temperature time series at 06:00 for 2016 (no monthly anomalies).',
    'region': 'lat 2–16N, lon 32–45E',
    'time_of_day': '06:00',
    'year': '2016'
})

# Preview the dataset structure
temp_series_ds = temp_series_ds.drop_vars("batch")
temp_series_ds

In [ ]:
# Monthly z-scores without introducing a separate 'month' dimension
month_idx = temp_series_ds['datetime.month']
month_avg = temp_series_ds.groupby('datetime.month').mean('datetime')
month_std = temp_series_ds.groupby('datetime.month').std('datetime', ddof=1)

# Guard against zero std to avoid inf
safe_std = month_std.where(month_std > 0)

# Align monthly stats to each datetime via indexing by month values
anom_in_std = (temp_series_ds - month_avg.sel(month=month_idx)) / safe_std.sel(month=month_idx)

In [ ]:
# Mean absolute z-scores on error vs other days
error_mask = anom_in_std.datetime.isin(error_times)
anom_on_error_days = np.abs(anom_in_std.sel(datetime=anom_in_std.datetime[error_mask])).mean(dim="datetime")
anom_on_other_days = np.abs(anom_in_std.sel(datetime=anom_in_std.datetime[~error_mask])).mean(dim="datetime")

In [ ]:
# Generate paired bar chart: mean absolute Z-scores on error days vs other days
import numpy as np
import matplotlib.pyplot as plt

# Base scalar variables (no pressure level)
base_vars = ['2m_temperature_era5', '2m_temperature_gc', 'skin_temperature_era5']

# Pressure levels for ERA5 temperature
plevels = temp_series_ds['temperature_era5'].pressure_level.values
level_labels = [f'temperature_era5_{int(np.round(float(lvl)))}hPa' for lvl in plevels]

# Build labels
labels = base_vars + level_labels

# Collect values (convert to Python floats)
error_vals = [float(anom_on_error_days[v].item()) for v in base_vars]
error_vals += [float(anom_on_error_days['temperature_era5'].sel(pressure_level=lvl).item()) for lvl in plevels]

other_vals = [float(anom_on_other_days[v].item()) for v in base_vars]
other_vals += [float(anom_on_other_days['temperature_era5'].sel(pressure_level=lvl).item()) for lvl in plevels]

# X positions
x = np.arange(len(labels))
width = 0.4

fig, ax = plt.subplots(figsize=(max(12, len(labels) * 0.6), 6))
ax.bar(x - width/2, error_vals, width=width, label='Error Days', color='#d62728')
ax.bar(x + width/2, other_vals, width=width, label='Other Days', color='#1f77b4')

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('Mean Absolute Z-Score')
ax.set_title('Mean Absolute Z-Scores on Error Days vs Other Days')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.3)

fig.tight_layout()
plt.show()

# Does the error also appear at 4am, 5am etc?

In [ ]:
error_data_6am = xr.open_mfdataset(
    [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/errors_2016{day.strftime('%m%d')}-06_n0.nc" for day in pd.date_range("2016-01-01", "2016-12-31")],
    concat_dim="datetime",
    combine="nested",
    preprocess=preprocess,
)

error_6am_timeseries = average_area("2016-01-01", "2016-12-31", 6, error_data_6am["2m_temperature"]).load()

error_times = error_6am_timeseries.datetime[abs(error_6am_timeseries) > 1]

In [ ]:
plt.figure(figsize=(12, 6))

era5_data = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/localised_2mt_2016_more_times.nc"))

times = average_area("2016-01-01", "2016-12-31", 6, era5_data["t2m"]).datetime
for hr in [4,6,8]:
    plt.plot(times, average_area("2016-01-01", "2016-12-31", hr, era5_data["t2m"]), label=f"Average 2m temperature at {hr}am")

plt.vlines(error_times, ymin=290, ymax=306, color='red', linestyle='--')

plt.legend()


In [ ]:
fig, axs = plt.subplots(5, 1, figsize=(12, 20), sharex=True)

for i, ax in enumerate(axs):
    axs[i].plot(times, average_area("2016-01-01", "2016-12-31", i+4, era5_data["t2m"]), label=f"ERA5 2m Temperature at {i+4}am")
    axs[i].set_title(f"ERA5 2m Temperature at {i+4}am")
    axs[i].set_ylabel("Temperature (K)")

    for error_time in error_times:
        axs[i].axvline(error_time.to_numpy(), color='red', linestyle='--', alpha=0.5)



# Satellite data

## Meteosat data


In [ ]:
base_dir = "/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/meteosat/ORD62079/"

files_6am = [os.path.join(base_dir, file) for file in os.listdir(base_dir) if file.endswith(".nc") and file[13:15]=="06"]

meteosat_data = xr.open_mfdataset(
    files_6am,
    combine="nested",
    concat_dim="time"
).rename({"time": "datetime"}).set_index({"datetime": "datetime"}).sortby("datetime")

meteosat_timeseries = average_area("2016-01-01", "2016-12-31", 6, meteosat_data["LST_PMW"]).load()
meteosat_unc_timeseries = average_area("2016-01-01", "2016-12-31", 6, meteosat_data["LSTERROR_PMW"]).load()


In [ ]:
era5_data = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/localised_2mt_2016.nc"))
era5_timeseries = average_area("2016-01-01", "2016-12-31", 6, era5_data["t2m"])

In [ ]:
gc_data = xr.open_mfdataset([f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/pred_2016{day.strftime('%m%d')}-06_n0.nc" for day in pd.date_range("2016-01-01", "2016-12-31")],
    concat_dim="datetime",
    combine="nested",
    preprocess=preprocess,
).rename({"time": "datetime"})
gc_data = gc_data.set_index({"datetime": "datetime"})

gc_timeseries = average_area("2016-01-01", "2016-12-31", 6, gc_data["2m_temperature"]).load()

In [ ]:
error_days = era5_timeseries.datetime[np.abs(era5_timeseries - gc_timeseries) > 1]

fig, axs = plt.subplots(6, 1, figsize=(10, 18))

axs[0].plot(meteosat_timeseries.datetime, meteosat_timeseries)
axs[0].set_title("Meteosat Surface Temperature")
axs[0].set_ylabel("Temperature (K)")

axs[1].plot(meteosat_unc_timeseries.datetime, meteosat_unc_timeseries)
axs[1].set_title("Meteosat Surface Temperature Uncertainty")
axs[1].set_ylabel("Temperature (K)")

axs[2].plot(era5_timeseries.datetime, era5_timeseries)
axs[2].set_title("ERA5 2m Temperature")
axs[2].set_ylabel("Temperature (K)")

axs[3].plot(gc_timeseries.datetime, gc_timeseries)
axs[3].set_title("Graphcast 2m Temperature")
axs[3].set_ylabel("Temperature (K)")

axs[4].plot(meteosat_timeseries.datetime, meteosat_timeseries-era5_timeseries)
axs[4].set_title("Meteosat Surface Temperature - ERA5 2m Temperature")
axs[4].set_ylabel("Temperature (K)")

axs[5].plot(meteosat_timeseries.datetime, meteosat_timeseries-gc_timeseries)
axs[5].set_title("Meteosat Surface Temperature - Graphcast 2m Temperature")
axs[5].set_ylabel("Temperature (K)")

for i, ax in enumerate(axs):
    for error_day in error_days:
        ax.axvline(error_day.to_numpy(), color='red', linestyle='--', alpha=0.5)
    if i != len(axs) - 1:
        ax.set_xticklabels([])


In [ ]:
error_days = era5_timeseries.datetime[np.abs(era5_timeseries - gc_timeseries) > 1]

fig, axs = plt.subplots(3, 1, figsize=(10, 12))

axs[0].plot(meteosat_timeseries.datetime, meteosat_timeseries)
axs[0].set_title("Meteosat Surface Temperature")
axs[0].set_ylabel("Temperature (K)")

axs[1].plot(era5_timeseries.datetime, era5_timeseries)
axs[1].set_title("ERA5 2m Temperature")
axs[1].set_ylabel("Temperature (K)")

axs[2].plot(gc_timeseries.datetime, gc_timeseries)
axs[2].set_title("Graphcast 2m Temperature")
axs[2].set_ylabel("Temperature (K)")

for i, ax in enumerate(axs):
    for error_day in error_days:
        ax.axvline(error_day.to_numpy(), color='red', linestyle='--', alpha=0.5)
    if i != len(axs) - 1:
        ax.set_xticklabels([])


In [ ]:
# find number of missing values in meteosat data
missing_meteosat = meteosat_timeseries.isnull()


missing_meteosat_days = meteosat_timeseries.datetime[missing_meteosat]

print(f"Number of missing values in Meteosat data: {int(missing_meteosat.sum())}")
print(f"Number of error days: {len(error_days)}")
print(f"Number of missing meteosat days that are also error days: {len(np.intersect1d(missing_meteosat_days, error_days))}")

In [ ]:
missing_meteosat_days

## CEDA data

In [ ]:
import xarray_regrid

ds = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/ceda_lst_data/20160101.nc")

ds['lst'] = ds['lst'].where(ds['lst'] <= 400)

  # Set unrealistic values to NaN

In [ ]:
ds["lst"].coarsen(lat = 100, lon=100).mean().plot()

## AQUA-AIRS

{"box":[30,0,50,20],"crop":false,"start":"2016-01-01T00:00:00Z","end":"2016-07-01T23:59:59Z","agent":"SUBSET_AIRS_L1L2","role":"subset","data":[{"datasetId":"AIRS2RET_7.0","variable":"TSurfStd"},{"datasetId":"AIRS2RET_7.0","variable":"TSurfStdErr"},{"datasetId":"AIRS2RET_7.0","variable":"TSurfAir"},{"datasetId":"AIRS2RET_7.0","variable":"TSurfAirErr"}],"sessionId":"689dc9809c7d1fba5b3c4343","format":"HDF"}

Job ID: https://disc.gsfc.nasa.gov/api/jobs/args/689ddd7490ebb8f82e3a5ae2

It seems like AQUA_AIRS never passes over our region around 6am, but observations could still be used to pin

In [ ]:
ds = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/meteosat/aqua_airs/AIRS.2016.01.01.100.L2.SUBS2RET.v7.0.1.0.G20085090023.hdf", engine="netcdf4")

origin = pd.to_datetime("1993-01-01")

(origin + pd.to_timedelta(ds["Time"].values.max(), unit="s")).hour

In [ ]:
def get_airs_time(ds):
    origin = pd.to_datetime("1993-01-01")
    time = origin + pd.to_timedelta(ds["Time"].values.mean(), unit="s")
    return time


aqua_datasets = []
aqua_dates = []
for file in os.listdir("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/meteosat/aqua_airs/"):
    if file.endswith(".hdf"):
        ds = xr.open_dataset(os.path.join("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/meteosat/aqua_airs/", file), engine="netcdf4")
        time = get_airs_time(ds)
        aqua_datasets.append(ds)
        aqua_dates.append(time)

In [ ]:
def process_airs_ds(ds):
    ny, nx = ds["TSurfStd"].shape

    new = xr.Dataset(
        {
            "Ts": (("y", "x"), ds["TSurfStd"].values, {
                "long_name": "2 metre surface temperature",
                "standard_name": "surface_temperature",
                "units": "K"
            }),
            "TsErr": (("y", "x"), ds["TSurfStdErr"].values, {
                "long_name": "2 metre surface temperature uncertainty",
                "standard_name": "surface_temperature_uncertainty",
                "units": "K"
            }),
            "Tair": (("y", "x"), ds["TSurfAir"].values, {
                "long_name": "2 metre air temperature",
                "standard_name": "air_temperature",
                "units": "K"
            }),
            "TairErr": (("y", "x"), ds["TSurfAirErr"].values, {
                "long_name": "2 metre air temperature uncertainty",
                "standard_name": "air_temperature_uncertainty",
                "units": "K"
            })

        },
        coords={
            "lat": (("y", "x"), ds["Latitude"].values, {
                "standard_name": "latitude",
                "units": "degrees_north"
            }),
            "lon": (("y", "x"), ds["Longitude"].values, {
                "standard_name": "longitude",
                "units": "degrees_east"
            })
        }
    )

    return new

def cover_frac(latvals, lonvals, minlon, maxlon, minlat, maxlat):
    inside_lon = (lonvals>=minlon) & (lonvals<=maxlon)
    inside_lat = (latvals>=minlat) & (latvals<=maxlat)

    inside = inside_lon & inside_lat

    dphi = np.gradient(np.deg2rad(latvals))[0]
    dlambda = np.gradient(np.deg2rad(lonvals))[1]
    phi = np.deg2rad(latvals)

    rectangle_area = (np.deg2rad(maxlon) - np.deg2rad(minlon)) * (np.sin(np.deg2rad(maxlat)) - np.sin(np.deg2rad(minlat)))
    cell_areas = np.abs(dphi * dlambda) * np.cos(phi)

    inside_areas = np.where(inside, cell_areas, 0)

    return inside_areas.sum() / rectangle_area

def average_inside(latvals, lonvals, minlon, maxlon, minlat, maxlat, tvals):
    inside_lon = (lonvals>=minlon) & (lonvals<=maxlon)
    inside_lat = (latvals>=minlat) & (latvals<=maxlat)
    inside = inside_lon & inside_lat
    if not inside.any():
        return np.nan
    return tvals[inside].mean()

aqua_datasets_processed = [process_airs_ds(ds) for ds in aqua_datasets]
minlon, maxlon = (32, 45)
minlat, maxlat = (2, 16)
cover_fractions = [cover_frac(ds['lat'], ds['lon'], minlon, maxlon, minlat, maxlat) for ds in aqua_datasets_processed]
t_avgs = [average_inside(ds['lat'], ds['lon'], minlon, maxlon, minlat, maxlat, ds['Ts'].values) for ds in aqua_datasets_processed]

# take into account that there could be multiple passes covering different parts

cover_fraction_dataset = xr.Dataset(
    {
        "cover_fraction": (("time"), cover_fractions)
    },
    {
        "time": (("time"), [d for d in aqua_dates])
    }
)

t_avg_dataset = xr.Dataset(
    {
        "t_avg": (("time"), t_avgs)
    },
    {
        "time": (("time"), [d for d in aqua_dates])
    }
)


In [ ]:
# sum all morning and evening occurances
morning_cfd = cover_fraction_dataset.sel(time=cover_fraction_dataset.time.dt.hour <= 12)
evening_cfd = cover_fraction_dataset.sel(time=cover_fraction_dataset.time.dt.hour > 12)
morning_total_cover = morning_cfd.groupby(morning_cfd.time.dt.date).sum()
evening_total_cover = evening_cfd.groupby(evening_cfd.time.dt.date).sum()

morning_temps = t_avg_dataset.sel(time=t_avg_dataset.time.dt.hour <= 12)
evening_temps = t_avg_dataset.sel(time=t_avg_dataset.time.dt.hour > 12)
morning_temps = morning_temps.groupby(morning_temps.time.dt.date).mean().sel(date = morning_total_cover.date[morning_total_cover["cover_fraction"] > 0.8])
evening_temps = evening_temps.groupby(evening_temps.time.dt.date).mean()

In [ ]:
error_6am_timeseries = average_area("2016-01-01", "2016-12-31", 6, error_data_6am["2m_temperature"]).load()

error_dates = error_6am_timeseries.datetime[error_6am_timeseries < -1]

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(morning_total_cover.date, morning_total_cover["cover_fraction"])
ax1.set_ylabel("Fraction of area covered by satellite observations")
ax1.set_title("Daily morning coverage of area by AQUA - Atmospheric Infrared Sounder (AIRS)")
# ax2 = ax1.twinx()
# ax2.plot(morning_temps.date, morning_temps["t_avg"], label="Morning Average Temperature", color="orange")
for date in [d for d in error_dates if d.dt.month<7]:
    ax1.axvline(x=date.to_numpy(), color="red", linestyle="--")
# plt.plot(evening_total_cover.date, evening_total_cover["cover_fraction"], label="Evening")


In [ ]:
i = 36

start, end = 34,43
n = end-start

fig, axs = plt.subplots(int(np.ceil(n/5)), 5, figsize=(15, 10), subplot_kw={'projection': ccrs.Robinson()})

for i in range(n):
    ds = aqua_datasets_processed[start+i]
    proj_map = ccrs.Robinson()          # or ccrs.PlateCarree(), etc.
    data_crs = ccrs.PlateCarree()       # because lat/lon are geographic

    ax = axs.flatten()[i]

    # fig, ax = plt.subplots(subplot_kw={'projection': proj_map}, figsize=(10,5))
    mesh = ax.pcolormesh(
        ds['lon'], ds['lat'], ds['Ts'],
        transform=data_crs,
        cmap='coolwarm',
        shading='auto'   # lets matplotlib decide; can use 'nearest' if you want blocky pixels
    )
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.5)
    ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.4, linestyle='--')
    # cb = fig.colorbar(mesh, ax=ax, shrink=0.7, pad=0.05)
    # cb.set_label('Temperature (K)')
    ax.set_title(f"{aqua_dates[start+i].strftime('%Y-%m-%dT%H:%M')}, Frac: {cover_frac(ds['lat'], ds['lon'], minlon, maxlon, minlat, maxlat):.2f}")
    ax.set_extent([0,50,-10,50])
plt.tight_layout()
plt.show()


## METOP IASI

In [ ]:
ds = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/meteosat/metop_iasi/W_XX-EUMETSAT-Darmstadt,HYPERSPECT+SOUNDING,METOPA+SO2+IASI_C_EUMP_20160102054154Z_20160102072354Z_eps_r_l2_0100.nc")

ds

In [ ]:
iasi_datasets = []
iasi_dates = []

iasi_path = "/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/meteosat/metop_iasi"

for file in os.listdir(iasi_path):
    if file.endswith(".nc"):
        ds = xr.open_dataset(os.path.join(iasi_path, file))

In [ ]:
ds.attrs

In [ ]:
iasi_ds = xr.open_mfdataset(
    [os.path.join(iasi_path, file) for file in os.listdir(iasi_path)[:10] if file.endswith(".nc")],
    combine="nested",
    concat_dim="time"
)